# 4. Tabulations

Turns each final long file into a readable cross-tab workbook: **one sheet per
indicator**, years across the columns, a hierarchical row index with merged
labels, and sourced values marked with a superscript that resolves to a numbered
footnote list at the bottom of the sheet.

```
COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx   ->   tabulations\<Chapter>_tabulations_EN.xlsx
COMPENDIUM-ARAB SOCIETY\<Chapter>_AR.xlsx   ->   tabulations\<Chapter>_tabulations_AR.xlsx
```

**Run this last.** It reads the final files, so it picks up both the merged
English-questionnaire rows (notebook 2) and the calculated indicators - the sex
ratio and the age-group percentages - which is the whole reason it comes after
the calculations rather than before.

## Known trade-off

A cell carrying a footnote marker becomes a rich-text **string**, not a live
number. Sums and charts will not work directly on those sheets. If you need the
figures to stay numeric, the long file is the place to compute from - or say so
and a plain no-superscript variant can be generated alongside.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths, chapters, and the row hierarchy.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\raffi\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\raffi\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Every long file lives here - one folder, both languages. The _AR / _EN suffix
# is already in each filename, so a folder per language only meant two places to
# look.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"
TABULATIONS_PATH = COMPENDIUM_PATH / "tabulations"

# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None
LANGUAGES = ["EN", "AR"]

# Declared once in English; translated per language further down, so there is
# never a second Arabic list to keep in step.
INDICATOR_COLUMN = "Indicator"
YEAR_COLUMN = "Year"
VALUE_COLUMN = "Value"
SOURCE_COLUMN = "Source"

# The outer row levels, in order - these open every sheet that has them.
ROW_COLUMNS = ["Country", "Sex", "Nationality"]

# Candidate inner levels. EVERY one of these that carries data for a given
# indicator becomes a nested row level for that sheet, innermost last. An
# indicator may have none (a plain Country x Year table) or several - causes of
# death carries both the ICD classification and the specific cause within it.
BREAKDOWN_COLUMNS = [
    "Area", "Age Group", "Marital status",
    "International classification for causes of death", "Causes of death",
    "Education level", "Educational sector", "Quintile",
    "Main occupation", "Institutional sector", "Economic activity",
    "Employment status", "Reasons for inactivity",
    "Type of living quarter", "Tenure of housing unit",
    "Source of water supply", "Source of Lighting",
    "Types of sewage disposal system", "Types of products/services",
]

# Shown when a row has no value for one of the row levels. It must be a visible
# label, never a blank - see build_sheet() for why.
MISSING_LABEL = {"EN": "(not specified)", "AR": "(غير محدد)"}


def discover_chapters():
    """Chapters with a final file in either language."""
    names = set()
    for language in LANGUAGES:
        suffix = f"_{language}.xlsx"
        for path in LONG_FILES_PATH.glob(f"*{suffix}"):
            if path.name.endswith("_EN_questionnaires.xlsx"):
                continue        # an input to notebook 2, not a final file
            names.add(path.name[: -len(suffix)])
    return sorted(names)


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever is ready to tabulate."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters ready to tabulate: {found}")
    if not found:
        logger.warning("No final files found - run notebooks 1-3 first.")
    return found


# ---------------------------------------------------------------------------
# Everything the pipeline finds wrong with the SOURCE DATA is collected here,
# from all four notebooks. Each owns a section and rewrites only its own, so the
# file always reflects the latest run of each step whatever order they ran in.
INCONSISTENCY_LOG_PATH = COMPENDIUM_PATH / "pipeline_inconsistencies.txt"


def save_inconsistencies(section, records):
    """Write this notebook's findings into the shared file, replacing its own
    section. `records` is a list of dicts; whichever of the locating fields are
    present are printed above each detail line, so a finding can be traced back
    to the exact country, indicator and year it came from."""
    marker = f"### {section} ###"
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")

    # In the order they help you narrow down a row.
    WHERE = ["chapter", "country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    body = [marker, f"    last run {stamp}", ""]
    if not records:
        body += ["    Nothing found.", ""]
    else:
        frame = pd.DataFrame(records)
        for kind, group in frame.groupby("kind", sort=False):
            body.append(f"  {kind.upper()}  ({len(group)})")
            for _, row in group.iterrows():
                def show(value):
                    # A record without a year forces that column to float, so
                    # 2010 would otherwise print as "2010.0".
                    if isinstance(value, float) and float(value).is_integer():
                        return str(int(value))
                    return str(value)
                where = " · ".join(
                    show(row[f]) for f in WHERE
                    if f in row and pd.notna(row[f]) and str(row[f]) != "")
                body.append(f"      {where}" if where else "      -")
                body.append(f"          {row['detail']}")
            body.append("")

    section_text = "\n".join(body)

    header = [
        "PIPELINE INCONSISTENCIES",
        "=" * 78,
        "",
    ]

    # Read what is already there and split it into sections, so this one can
    # replace its own and the file be rebuilt in step order. Appending instead
    # left the sections in whatever order the notebooks last ran, which reads as
    # though steps had been skipped.
    sections = {}
    if INCONSISTENCY_LOG_PATH.exists():
        existing = INCONSISTENCY_LOG_PATH.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    INCONSISTENCY_LOG_PATH.write_text(
        "\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n", encoding="utf-8")
    return INCONSISTENCY_LOG_PATH, len(records)


## Column names in either language

The row hierarchy is declared once in English and looked up in the dictionary
for the Arabic file, so there is no second list to keep in step.


In [ ]:
"""
CELL: columns_for() - the same column names, spelled for either language.
"""


def load_column_names():
    """{English column name: Arabic column name}, from the dictionary."""
    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    pairs = dictionary[["col_en", "col_ar"]].dropna().drop_duplicates()
    return {str(en).strip(): str(ar).strip() for en, ar in pairs.itertuples(index=False)}


ENGLISH_TO_ARABIC_COLUMNS = load_column_names()


def column_in(english_name, language):
    """One column name, spelled for the file being read. Looking the Arabic name
    up here means the two lists can never drift apart."""
    if language == "EN":
        return english_name
    return ENGLISH_TO_ARABIC_COLUMNS.get(english_name, english_name)


def columns_for(language):
    """Every column name this notebook needs, for one language."""
    return {
        "indicator": column_in(INDICATOR_COLUMN, language),
        "year": column_in(YEAR_COLUMN, language),
        "value": column_in(VALUE_COLUMN, language),
        "source": column_in(SOURCE_COLUMN, language),
        "rows": [column_in(c, language) for c in ROW_COLUMNS],
        "breakdowns": [column_in(c, language) for c in BREAKDOWN_COLUMNS],
        "missing": MISSING_LABEL[language],
    }


logger.info(f"Column names loaded for {LANGUAGES}")


## Building one sheet

Three things here are less obvious than they look, and each was a bug once.

**An indicator may have no breakdown column, or several.** Taking only the first
would silently collapse the rest under `aggfunc="first"` - on Population that
meant losing 63% of the rows. Every populated breakdown becomes a nested level.

**A missing dimension value must be labelled, never left blank.** This layout
blanks a repeated label to show it continues from the row above, so an empty
cell reads as "same as above". A row genuinely missing a value would be absorbed
into the group above it and misattribute real numbers.

**Row-label merging has to cascade.** If an outer level changes between two rows,
every inner level counts as changed too, even where its own value repeats.
Compute all the changed-flags against the previous row *before* merging, then
update - updating level by level mid-loop makes the outer check see stale values
and produces wrong merges that look fine on the first rows and break deeper down.


In [ ]:
"""
CELL: build_sheet() - one indicator's cross-tab, with merged row labels and footnotes.
"""
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.utils import get_column_letter

HEADER_FONT = Font(bold=True)
THIN = Side(style="thin", color="B0B0B0")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
SUPERSCRIPT_FONT = InlineFont(vertAlign="superscript", sz="9")


def sanitize_sheet_name(name, used):
    """Excel caps sheet names at 31 characters and forbids []:*?/\\ - and two
    indicators can truncate to the same thing, so de-duplicate as well."""
    clean = re.sub(r"[\[\]\:\*\?/\\]", "", str(name)).replace("\xa0", " ").strip()
    base = clean[:31] if clean else "Sheet"
    candidate, n = base, 2
    while candidate in used:
        suffix = f" ({n})"
        candidate = base[: 31 - len(suffix)] + suffix
        n += 1
    used.add(candidate)
    return candidate


def populated_breakdowns(sub, breakdown_columns):
    """Every candidate breakdown column that actually carries data for this
    indicator, in the order given - all of them become nested row levels."""
    return [c for c in breakdown_columns if c in sub.columns and sub[c].notna().any()]


def to_display_str(value):
    """Clean numeric formatting where possible, plain string otherwise."""
    try:
        number = float(value)
    except (ValueError, TypeError):
        return str(value)
    return str(int(number)) if number.is_integer() else f"{number:.1f}"


def has_value(x):
    if pd.isna(x):
        return False
    return not (isinstance(x, str) and x.strip() == "")


def build_sheet(ws, sub, row_columns, breakdowns, year_column, value_column,
                source_column, missing_label):
    full_row_columns = list(row_columns) + list(breakdowns)

    sub = sub.copy()
    for c in full_row_columns:
        sub[c] = sub[c].fillna("").astype(str).str.strip()

    # Drop a level that is empty for EVERY row - it would render as a headed but
    # completely blank column.
    full_row_columns = [c for c in full_row_columns if (sub[c] != "").any()]
    if not full_row_columns:
        ws["A1"] = "No row dimensions available for this indicator."
        return

    # Remaining blanks get a visible marker. See the notes above.
    for c in full_row_columns:
        sub[c] = sub[c].replace("", missing_label)

    grouped = (sub.groupby(full_row_columns + [year_column], dropna=False, observed=True)
               .agg(**{value_column: (value_column, "first"),
                       source_column: (source_column, "first")})
               .reset_index())

    years = sorted(y for y in grouped[year_column].unique()
                   if grouped.loc[grouped[year_column] == y, value_column].apply(has_value).any())
    if not years:
        ws["A1"] = "No data available for this indicator."
        return

    values = grouped.pivot_table(index=full_row_columns, columns=year_column,
                                 values=value_column, aggfunc="first").reindex(columns=years)
    sources = grouped.pivot_table(index=full_row_columns, columns=year_column,
                                  values=source_column, aggfunc="first") \
                     .reindex(index=values.index, columns=values.columns)

    keep = values.apply(lambda col: col.map(has_value)).any(axis=1).to_numpy()
    values = values.loc[keep].sort_index()
    sources = sources.loc[keep].reindex(values.index)
    if values.empty:
        ws["A1"] = "No data available for this indicator."
        return

    n_levels, n_years = len(full_row_columns), len(years)
    first_row = 2
    last_row = first_row + len(values) - 1

    for level, name in enumerate(full_row_columns, start=1):
        cell = ws.cell(row=1, column=level, value=name)
        cell.font, cell.border = HEADER_FONT, BORDER
    for j, year in enumerate(years):
        cell = ws.cell(row=1, column=n_levels + 1 + j,
                       value=int(year) if float(year).is_integer() else year)
        cell.font, cell.border = HEADER_FONT, BORDER
        cell.alignment = Alignment(horizontal="center")

    footnotes, order = {}, []

    def footnote_number(source):
        if pd.isna(source) or str(source).strip() == "":
            return None
        key = str(source).strip()
        if key not in footnotes:
            footnotes[key] = len(order) + 1
            order.append(key)
        return footnotes[key]

    # Row labels, with cascading vertical merges.
    tuples = list(values.index)
    level_start = {lvl: first_row for lvl in range(n_levels)}
    previous = [None] * n_levels

    for i, tup in enumerate(tuples):
        r = first_row + i
        row_values = list(tup) if n_levels > 1 else [tup]

        changed, cascade = [False] * n_levels, False
        for lvl in range(n_levels):
            if i == 0 or row_values[lvl] != previous[lvl]:
                cascade = True
            changed[lvl] = cascade

        for lvl in range(n_levels):
            if changed[lvl]:
                start = level_start[lvl]
                if i != 0 and r - 1 > start:
                    ws.merge_cells(start_row=start, start_column=lvl + 1,
                                   end_row=r - 1, end_column=lvl + 1)
                cell = ws.cell(row=r, column=lvl + 1, value=row_values[lvl])
                cell.alignment = Alignment(vertical="center")
                cell.border = BORDER
                level_start[lvl] = r
        previous = row_values

    for lvl in range(n_levels):
        start = level_start[lvl]
        if last_row > start:
            ws.merge_cells(start_row=start, start_column=lvl + 1,
                           end_row=last_row, end_column=lvl + 1)
    for r in range(first_row, last_row + 1):
        for lvl in range(n_levels):
            ws.cell(row=r, column=lvl + 1).border = BORDER

    for i in range(len(tuples)):
        r = first_row + i
        for j in range(n_years):
            cell = ws.cell(row=r, column=n_levels + 1 + j)
            cell.border = BORDER
            cell.alignment = Alignment(horizontal="center")
            value = values.iloc[i, j]
            if not has_value(value):
                continue
            number = footnote_number(sources.iloc[i, j])
            text = to_display_str(value)
            cell.value = (CellRichText(text, TextBlock(SUPERSCRIPT_FONT, str(number)))
                          if number else text)

    footnote_row = last_row + 2
    for source, number in sorted(footnotes.items(), key=lambda kv: kv[1]):
        cell = ws.cell(row=footnote_row, column=1, value=f"{number}  {source}")
        cell.font = Font(size=9, italic=True)
        footnote_row += 1

    for lvl in range(n_levels):
        ws.column_dimensions[get_column_letter(lvl + 1)].width = 20
    for j in range(n_years):
        ws.column_dimensions[get_column_letter(n_levels + 1 + j)].width = 10
    ws.freeze_panes = ws.cell(row=first_row, column=n_levels + 1)


## `build_tabulation()`


In [ ]:
"""
CELL: build_tabulation() - one workbook per long file.
"""


def build_tabulation(chapter, language):
    """Reads <chapter>_<language>.xlsx and writes tabulations/<chapter>_tabulations_<language>.xlsx.
    Returns the number of sheets written, or 0 if there was nothing to read."""
    source_path = LONG_FILES_PATH / f"{chapter}_{language}.xlsx"
    if not source_path.exists():
        logger.info(f"  {chapter} {language}: no {source_path.name}, skipping")
        return 0

    names = columns_for(language)
    table = pd.read_excel(source_path, engine="openpyxl")

    required = [names["indicator"], names["year"], names["value"], names["source"]]
    missing = [c for c in required if c not in table.columns]
    if missing:
        logger.warning(f"  {chapter} {language}: missing column(s) {missing}, skipping")
        return 0

    row_columns = [c for c in names["rows"] if c in table.columns]
    indicators = table[names["indicator"]].dropna().unique()

    workbook = Workbook()
    workbook.remove(workbook.active)
    used_names = set()

    for indicator in indicators:
        sub = table[table[names["indicator"]] == indicator]
        breakdowns = populated_breakdowns(sub, names["breakdowns"])
        ws = workbook.create_sheet(title=sanitize_sheet_name(indicator, used_names))
        build_sheet(ws, sub, row_columns, breakdowns, names["year"], names["value"],
                    names["source"], names["missing"])

    TABULATIONS_PATH.mkdir(parents=True, exist_ok=True)
    out_path = TABULATIONS_PATH / f"{chapter}_tabulations_{language}.xlsx"
    workbook.save(out_path)
    logger.info(f"  {chapter} {language}: {len(indicators)} indicator sheet(s) "
                f"-> tabulations\\{out_path.name}")
    return len(indicators)


## Run - tabulate every chapter, both languages


In [ ]:
"""
CELL: Main run - a tabulation workbook for every chapter, in both languages.
"""
print(f"Reading  {LONG_FILES_PATH}\\<Chapter>_<LANG>.xlsx")
print(f"Writing  {TABULATIONS_PATH}\n")

built = {}
steps = len(chapters_to_process()) * len(LANGUAGES)
step = 0
for chapter in chapters_to_process():
    for language in LANGUAGES:
        step += 1
        bar = "#" * step + "-" * (steps - step)
        print(f"[{bar}] {step}/{steps}  {chapter} {language}")
        sheets = build_tabulation(chapter, language)
        if sheets:
            built[(chapter, language)] = sheets

print("\n" + "=" * 70)
print("TABULATIONS WRITTEN")
print("=" * 70)
if not built:
    print("None - run notebooks 1-3 first.")
else:
    for (chapter, language), sheets in sorted(built.items()):
        print(f"  tabulations\\{chapter}_tabulations_{language}.xlsx   {sheets:>3} sheet(s)")
    print(f"\n  {len(built)} workbook(s). Remember: cells carrying a footnote marker")
    print("  are rich-text strings, not live numbers.")


## Verify before delivering

Merged-cell logic for hierarchical row labels is the most error-prone part, so
check rather than assume. The cell below re-opens each workbook and asserts the
thing that actually matters: after forward-filling merged labels, **every data
row must have a unique combination of row labels**. A duplicate means the
cascade collapsed two genuinely different rows together.

It also reports sheets that came out empty, which usually means an indicator has
no usable rows rather than a bug.


In [ ]:
"""
CELL: records = []
verify_tabulations(record=records)
written, count = save_inconsistencies("4. TABULATIONS", records)
print(f"\n{count} inconsistency(ies) recorded in {written.name}") - check the merged row labels are sane.
"""
from openpyxl import load_workbook


def verify_tabulations(record=None):
    """Re-open every workbook written above and check each sheet's row labels.

    Forward-fills the merged label columns and asserts each data row's label
    tuple is unique. The data block ends at the first fully empty row - after
    that come the footnotes, which are not data and must not be counted.
    """
    workbooks = sorted(TABULATIONS_PATH.glob("*_tabulations_*.xlsx"))
    if not workbooks:
        print("No tabulation workbooks found - run the cell above first.")
        return

    for path in workbooks:
        workbook = load_workbook(path)
        empty_sheets, duplicate_sheets = [], []

        for ws in workbook.worksheets:
            first_cell = ws["A1"].value
            if first_cell and "No " in str(first_cell):
                empty_sheets.append(ws.title)
                continue

            header = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
            n_levels = sum(1 for h in header if h is not None and not str(h).isdigit())

            seen, duplicates = set(), 0
            last = [None] * n_levels
            for row in ws.iter_rows(min_row=2):
                if all(c.value is None for c in row):
                    break                      # blank separator: end of the data
                labels = []
                for i, cell in enumerate(row[:n_levels]):
                    if cell.value is not None:
                        last[i] = cell.value
                    labels.append(last[i])
                key = tuple(labels)
                if key in seen:
                    duplicates += 1
                seen.add(key)
            if duplicates:
                duplicate_sheets.append((ws.title, duplicates))
                if record is not None:
                    record.append({
                        "kind": "tabulation row labels collide",
                        "detail": f"{path.name} · {ws.title}: {duplicates} row(s) share "
                                  f"a label combination, so two different rows read as one",
                    })

        workbook.close()
        status = "OK" if not duplicate_sheets else "PROBLEM"
        print(f"{status:<8} {path.name}: {len(workbook.sheetnames)} sheet(s), "
              f"{len(empty_sheets)} empty, {len(duplicate_sheets)} with duplicate row labels")
        for title, n in duplicate_sheets[:5]:
            print(f"             {title}: {n} duplicate row(s)")


records = []
verify_tabulations(record=records)
written, count = save_inconsistencies("4. TABULATIONS", records)
print(f"\n{count} inconsistency(ies) recorded in {written.name}")
